In [ ]:
# Install all required packages
!pip install skl2onnx onnxruntime onnx librosa joblib scikit-learn numpy -q
print('✅ All packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 74.7 MB/s eta 0:00:00
✅ All packages installed!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted!')

Mounted at /content/drive
✅ Google Drive mounted!


In [ ]:
import os
import numpy as np
import joblib

# ─────────────────────────────────────────────────────────
# 🔧 CONFIGURE YOUR PATH HERE
# ─────────────────────────────────────────────────────────
MODEL_DIR  = '/content/drive/MyDrive/respiratory_disease_classification-master/data/outputs/models'
OUTPUT_DIR = '/content/drive/MyDrive/respiratory_disease_classification-master/data/outputs'

# Load the trained artifacts saved in the original notebook
knn_model     = joblib.load(os.path.join(MODEL_DIR, 'best_knn_mfcc_mean_std.joblib'))
scaler        = joblib.load(os.path.join(MODEL_DIR, 'scaler.joblib'))
label_encoder = joblib.load(os.path.join(MODEL_DIR, 'label_encoder.joblib'))

print('✅ Loaded:')
print(f'  KNN model      : {type(knn_model).__name__}')
print(f'  Scaler         : {type(scaler).__name__}')
print(f'  Label Encoder  : classes = {list(label_encoder.classes_)}')
print(f'  KNN n_neighbors: {knn_model.n_neighbors}')
print(f'  KNN metric     : {knn_model.metric}')

✅ Loaded:
  KNN model      : KNeighborsClassifier
  Scaler         : StandardScaler
  Label Encoder  : classes = ['Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'Pneumonia', 'URTI']
  KNN n_neighbors: 1
  KNN metric     : euclidean


In [ ]:
# Feature column definitions (must match original training notebook exactly)
mfcc_mean_cols = [f'mfccs_mean_{i}' for i in range(13)]
mfcc_std_cols  = [f'mfccs_std_{i}'  for i in range(13)]
MFCC_BEST_COLS = mfcc_mean_cols + mfcc_std_cols  # 26 features used by best model

# All 80 feature columns in exact order (matches scaler training order)
ALL_80_COLS = (
    ['chroma_cens_mean', 'chroma_cens_std', 'chroma_cens_var',      # 3
     'mel_mean', 'mel_std', 'mel_var',                              # 3
     'mfcc_mean', 'mfcc_std', 'mfcc_var',                          # 3
     'mfcc_delta_mean', 'mfcc_delta_std', 'mfcc_delta_var',        # 3
     'cent_mean', 'cent_std', 'cent_var',                          # 3
     'spec_bw_mean', 'spec_bw_std', 'spec_bw_var',                 # 3
     'rolloff_mean', 'rolloff_std', 'rolloff_var',                  # 3
     'zcr_mean', 'zcr_std', 'zcr_var',                             # 3
     'harm_mean', 'harm_std', 'harm_var',                          # 3
     'perc_mean', 'perc_std', 'perc_var']                          # 3 = 30
    + [f'mfccs_mean_{i}' for i in range(13)]                       # 13
    + [f'mfccs_std_{i}'  for i in range(13)]                       # 13
    + [f'chroma_mean_{i}' for i in range(12)]                      # 12
    + [f'chroma_std_{i}'  for i in range(12)]                      # 12 = 80
)

# Index positions of the 26 MFCC features within the 80-feature scaler
MFCC_INDICES = [ALL_80_COLS.index(col) for col in MFCC_BEST_COLS]

print(f'Total feature columns : {len(ALL_80_COLS)}')
print(f'MFCC best cols        : {len(MFCC_BEST_COLS)}')
print(f'MFCC indices in scaler: {MFCC_INDICES[:5]} ... {MFCC_INDICES[-5:]}')

Total feature columns : 80
MFCC best cols        : 26
MFCC indices in scaler: [30, 31, 32, 33, 34] ... [51, 52, 53, 54, 55]


In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# ─────────────────────────────────────────────────────────
# Step 1: Extract scaler parameters for only MFCC features
# ─────────────────────────────────────────────────────────
# The original scaler was fit on all 80 features.
# We create a new scaler for just the 26 MFCC features.
mfcc_scaler = StandardScaler()
mfcc_scaler.mean_    = scaler.mean_[MFCC_INDICES]
mfcc_scaler.scale_   = scaler.scale_[MFCC_INDICES]
mfcc_scaler.var_     = scaler.var_[MFCC_INDICES]
mfcc_scaler.n_features_in_ = 26
mfcc_scaler.n_samples_seen_ = scaler.n_samples_seen_
mfcc_scaler.feature_names_in_ = None  # avoid name mismatch

print('✅ MFCC-specific scaler extracted')
print(f'   mean_[:5]  = {mfcc_scaler.mean_[:5]}')
print(f'   scale_[:5] = {mfcc_scaler.scale_[:5]}')

✅ MFCC-specific scaler extracted
   mean_[:5]  = [-352.6546253   114.52646879   46.94637539   29.01041856   28.98313125]
   scale_[:5] = [58.68377234 29.75148153 18.31115844 16.58639379  8.83415081]


In [ ]:
# ─────────────────────────────────────────────────────────
# Step 2: Build sklearn Pipeline (Scaler → KNN)
# ─────────────────────────────────────────────────────────
pipeline = Pipeline([
    ('scaler', mfcc_scaler),
    ('knn',    knn_model)
])

# Verify pipeline works with a dummy input
dummy_input = np.zeros((1, 26), dtype=np.float32)
_ = pipeline.predict(dummy_input)
print('✅ Pipeline verified with dummy input')

✅ Pipeline verified with dummy input


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# ─────────────────────────────────────────────────────────
# Step 3: Convert Pipeline to ONNX
# ─────────────────────────────────────────────────────────
# Input: 26 float32 features (MFCC mean_0..12, std_0..12)
initial_type = [('mfcc_features', FloatTensorType([None, 26]))]

onnx_model = convert_sklearn(
    pipeline,
    initial_types=initial_type,
    target_opset=17,
    options={'zipmap': False}   # return array probabilities, not dict
)

# Save ONNX model
ONNX_PATH = os.path.join(OUTPUT_DIR, 'respiratory_knn.onnx')
with open(ONNX_PATH, 'wb') as f:
    f.write(onnx_model.SerializeToString())

print(f'✅ ONNX model saved to: {ONNX_PATH}')
print(f'   File size: {os.path.getsize(ONNX_PATH) / 1024:.1f} KB')

✅ ONNX model saved to: /content/drive/MyDrive/respiratory_disease_classification-master/data/outputs/respiratory_knn.onnx
   File size: 202.1 KB


In [ ]:
import onnxruntime as rt
import pandas as pd

# ─────────────────────────────────────────────────────────
# Load ONNX model
# ─────────────────────────────────────────────────────────
sess = rt.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
input_name  = sess.get_inputs()[0].name
label_name  = sess.get_outputs()[0].name
prob_name   = sess.get_outputs()[1].name

print('ONNX Model Inputs:')
for inp in sess.get_inputs():
    print(f'  name={inp.name}, shape={inp.shape}, type={inp.type}')

print('\nONNX Model Outputs:')
for out in sess.get_outputs():
    print(f'  name={out.name}, shape={out.shape}, type={out.type}')

ONNX Model Inputs:
  name=mfcc_features, shape=[None, 26], type=tensor(float)

ONNX Model Outputs:
  name=label, shape=[None], type=tensor(int64)
  name=probabilities, shape=[None, 6], type=tensor(float)


In [ ]:
# ─────────────────────────────────────────────────────────
# Generate random test samples and compare predictions
# ─────────────────────────────────────────────────────────
np.random.seed(42)
N_TESTS = 100
test_inputs = np.random.randn(N_TESTS, 26).astype(np.float32)

# sklearn predictions (original model) — apply scaler then predict
sklearn_preds = pipeline.predict(test_inputs)

# ONNX predictions
onnx_preds_raw, onnx_probs = sess.run(
    [label_name, prob_name],
    {input_name: test_inputs}
)
onnx_preds = np.array(onnx_preds_raw)

# Compare
matches = np.sum(sklearn_preds == onnx_preds)
match_pct = matches / N_TESTS * 100

print(f'Predictions on {N_TESTS} random inputs:')
print(f'  sklearn predictions : {sklearn_preds[:10]}')
print(f'  ONNX    predictions : {onnx_preds[:10]}')
print(f'  Matches             : {matches}/{N_TESTS} ({match_pct:.1f}%)')

if match_pct == 100.0:
    print('\n🎉 PERFECT MATCH! ONNX model is 100% equivalent to sklearn model.')
else:
    print(f'\n⚠️  Mismatch on {N_TESTS - matches} samples. Check scaler indices.')

Predictions on 100 random inputs:
  sklearn predictions : [2 2 2 2 2 2 2 2 2 2]
  ONNX    predictions : [2 2 2 2 2 2 2 2 2 2]
  Matches             : 100/100 (100.0%)

🎉 PERFECT MATCH! ONNX model is 100% equivalent to sklearn model.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# ─────────────────────────────────────────────────────────
# Inspect ONNX output probabilities for a sample
# ─────────────────────────────────────────────────────────
sample = test_inputs[:1]
onnx_label, onnx_prob = sess.run([label_name, prob_name], {input_name: sample})

print('\nSample ONNX prediction:')
print(f'  Predicted class (encoded): {onnx_label[0]}')
print(f'  Decoded class label       : {label_encoder.classes_[onnx_label[0]]}')
print(f'  Class probabilities:')
for cls, prob in zip(label_encoder.classes_, onnx_prob[0]):
    bar = '█' * int(prob * 30)
    print(f'    {cls:<18}: {prob:.4f}  {bar}')


Sample ONNX prediction:
  Predicted class (encoded): 2
  Decoded class label       : COPD
  Class probabilities:
    Bronchiectasis    : 0.0000  
    Bronchiolitis     : 0.0000  
    COPD              : 1.0000  ██████████████████████████████
    Healthy           : 0.0000  
    Pneumonia         : 0.0000  
    URTI              : 0.0000  


In [ ]:
import librosa
import onnxruntime as rt
import numpy as np
import pandas as pd
from google.colab import files

# ─────────────────────────────────────────────────────────
# Audio constants (MUST match original notebook)
# ─────────────────────────────────────────────────────────
SR          = 16000
SEGMENT_LEN = 4 * SR

def truncate_or_pad(y, length=SEGMENT_LEN):
    if len(y) > length:  return y[:length]
    if len(y) < length:  return np.pad(y, (0, length - len(y)))
    return y

def extract_26_mfcc_features(y, sr=SR):
    """
    Extract only the 26 MFCC features used by the best model.
    This matches the original notebook's feature extraction exactly.
    Returns: numpy array of shape (26,)
    """
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    feats = {}
    for i in range(13):
        feats[f'mfccs_mean_{i}'] = float(np.mean(mfcc[i]))
    for i in range(13):
        feats[f'mfccs_std_{i}']  = float(np.std(mfcc[i]))
    return feats

print('✅ Helper functions defined')
print('📂 Please upload a .wav audio file below:')

✅ Helper functions defined
📂 Please upload a .wav audio file below:


In [15]:
# Upload audio file
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f'Uploaded: {file_name}')

# Load and preprocess
audio, _ = librosa.load(file_name, sr=SR)
audio     = truncate_or_pad(audio)
print(f'Audio shape: {audio.shape} (expected: {SEGMENT_LEN})')

# Extract 26 MFCC features
feat_dict = extract_26_mfcc_features(audio, sr=SR)
feat_arr  = np.array([list(feat_dict.values())], dtype=np.float32)  # shape (1, 26)
print(f'Feature array shape: {feat_arr.shape}')

# ─────────────────────────────────────────────────────────
# Run ONNX inference  (scaler + KNN baked inside)
# ─────────────────────────────────────────────────────────
sess_inf  = rt.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
in_name   = sess_inf.get_inputs()[0].name
lbl_name  = sess_inf.get_outputs()[0].name
prb_name  = sess_inf.get_outputs()[1].name

pred_label, pred_probs = sess_inf.run([lbl_name, prb_name], {in_name: feat_arr})

# Decode
decoded = label_encoder.classes_[pred_label[0]]
confidence = float(np.max(pred_probs[0])) * 100

print('\n' + '='*50)
print(f'🎯 ONNX Predicted Class : {decoded}')
print(f'   Confidence           : {confidence:.1f}%')
print('='*50)
print('\n📊 All class probabilities:')
for cls, p in sorted(zip(label_encoder.classes_, pred_probs[0]), key=lambda x: -x[1]):
    bar = '█' * int(p * 40)
    print(f'  {cls:<18}: {p*100:5.1f}%  {bar}')

Saving test1.wav to test1 (1).wav
Uploaded: test1 (1).wav
Audio shape: (64000,) (expected: 64000)
Feature array shape: (1, 26)

🎯 ONNX Predicted Class : Pneumonia
   Confidence           : 100.0%

📊 All class probabilities:
  Pneumonia         : 100.0%  ████████████████████████████████████████
  Bronchiectasis    :   0.0%  
  Bronchiolitis     :   0.0%  
  COPD              :   0.0%  
  Healthy           :   0.0%  
  URTI              :   0.0%  


In [16]:
# ─────────────────────────────────────────────────────────
# Cross-check: Same file with original sklearn model
# ─────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# Reproduce exact original notebook prediction pipeline
def extract_80_features(y, sr=SR):
    features = {}
    chroma_cens = librosa.feature.chroma_cens(y=y, sr=sr)
    features['chroma_cens_mean'] = np.mean(chroma_cens)
    features['chroma_cens_std']  = np.std(chroma_cens)
    features['chroma_cens_var']  = np.var(chroma_cens)
    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    features['mel_mean'] = np.mean(mel)
    features['mel_std']  = np.std(mel)
    features['mel_var']  = np.var(mel)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    features['mfcc_mean'] = np.mean(mfcc)
    features['mfcc_std']  = np.std(mfcc)
    features['mfcc_var']  = np.var(mfcc)
    mfcc_delta = librosa.feature.delta(mfcc)
    features['mfcc_delta_mean'] = np.mean(mfcc_delta)
    features['mfcc_delta_std']  = np.std(mfcc_delta)
    features['mfcc_delta_var']  = np.var(mfcc_delta)
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    features['cent_mean'] = np.mean(cent); features['cent_std'] = np.std(cent); features['cent_var'] = np.var(cent)
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features['spec_bw_mean'] = np.mean(spec_bw); features['spec_bw_std'] = np.std(spec_bw); features['spec_bw_var'] = np.var(spec_bw)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features['rolloff_mean'] = np.mean(rolloff); features['rolloff_std'] = np.std(rolloff); features['rolloff_var'] = np.var(rolloff)
    zcr = librosa.feature.zero_crossing_rate(y)
    features['zcr_mean'] = np.mean(zcr); features['zcr_std'] = np.std(zcr); features['zcr_var'] = np.var(zcr)
    harm, perc = librosa.effects.hpss(y)
    features['harm_mean'] = np.mean(harm); features['harm_std'] = np.std(harm); features['harm_var'] = np.var(harm)
    features['perc_mean'] = np.mean(perc); features['perc_std'] = np.std(perc); features['perc_var'] = np.var(perc)
    for i in range(13): features[f'mfccs_mean_{i}'] = np.mean(mfcc[i])
    for i in range(13): features[f'mfccs_std_{i}']  = np.std(mfcc[i])
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(12): features[f'chroma_mean_{i}'] = np.mean(chroma[i])
    for i in range(12): features[f'chroma_std_{i}']  = np.std(chroma[i])
    return features

mfcc_mean_cols_orig = [f'mfccs_mean_{i}' for i in range(13)]
mfcc_std_cols_orig  = [f'mfccs_std_{i}'  for i in range(13)]
MFCC_BEST_COLS_orig = mfcc_mean_cols_orig + mfcc_std_cols_orig

raw_feats = extract_80_features(audio, sr=SR)
df_all = pd.DataFrame([raw_feats])
scaled_all = scaler.transform(df_all)
df_scaled = pd.DataFrame(scaled_all, columns=df_all.columns)
feat_for_pred = df_scaled[MFCC_BEST_COLS_orig].values

sklearn_pred = knn_model.predict(feat_for_pred)
sklearn_decoded = label_encoder.classes_[sklearn_pred[0]]

print('\n' + '='*50)
print(f'Original sklearn  : {sklearn_decoded}')
print(f'ONNX model        : {decoded}')
match = '✅ MATCH!' if sklearn_decoded == decoded else '❌ MISMATCH'
print(f'Result            : {match}')
print('='*50)


Original sklearn  : Pneumonia
ONNX model        : Pneumonia
Result            : ✅ MATCH!


In [ ]:
import json

# Create label map: index → class name
label_map = {int(i): str(cls) for i, cls in enumerate(label_encoder.classes_)}
label_map_path = os.path.join(OUTPUT_DIR, 'label_map.json')
with open(label_map_path, 'w') as f:
    json.dump(label_map, f, indent=2)

# Also save MFCC feature indices (for React: which 26 cols to extract from 80)
feature_info = {
    'mfcc_best_cols': MFCC_BEST_COLS,
    'n_features': 26,
    'sr': 16000,
    'segment_len_sec': 4,
    'n_mfcc': 13,
    'classes': list(label_encoder.classes_)
}
feat_info_path = os.path.join(OUTPUT_DIR, 'model_feature_info.json')
with open(feat_info_path, 'w') as f:
    json.dump(feature_info, f, indent=2)

print('✅ label_map.json saved')
print(f'   Classes: {label_map}')
print(f'\n✅ model_feature_info.json saved')
print(json.dumps(feature_info, indent=2))

✅ label_map.json saved
   Classes: {0: 'Bronchiectasis', 1: 'Bronchiolitis', 2: 'COPD', 3: 'Healthy', 4: 'Pneumonia', 5: 'URTI'}

✅ model_feature_info.json saved
{
  "mfcc_best_cols": [
    "mfccs_mean_0",
    "mfccs_mean_1",
    "mfccs_mean_2",
    "mfccs_mean_3",
    "mfccs_mean_4",
    "mfccs_mean_5",
    "mfccs_mean_6",
    "mfccs_mean_7",
    "mfccs_mean_8",
    "mfccs_mean_9",
    "mfccs_mean_10",
    "mfccs_mean_11",
    "mfccs_mean_12",
    "mfccs_std_0",
    "mfccs_std_1",
    "mfccs_std_2",
    "mfccs_std_3",
    "mfccs_std_4",
    "mfccs_std_5",
    "mfccs_std_6",
    "mfccs_std_7",
    "mfccs_std_8",
    "mfccs_std_9",
    "mfccs_std_10",
    "mfccs_std_11",
    "mfccs_std_12"
  ],
  "n_features": 26,
  "sr": 16000,
  "segment_len_sec": 4,
  "n_mfcc": 13,
  "classes": [
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "Pneumonia",
    "URTI"
  ]
}
